# Réentraînement du DNN HDVR : v4 → v5 (Colab)

**Avant de lancer :** Exécution → Modifier le type d'exécution → GPU (T4) conseillé (le CPU fonctionne aussi).
Placez dans `MyDrive/nids/` : `dnn_v4.pt`, `scaler_v4.joblib`, `meta_v4.json`, `env_v4.json`, `classes.json`, `feature_cols.json`, et le dataset NFStream dans `MyDrive/nids/data/`.

Le dataset doit contenir : les 58 colonnes de `feature_cols.json`, une colonne d'étiquette et une colonne de temps en **millisecondes** (ex. `bidirectional_first_seen_ms`).

Le notebook reproduit le prétraitement et le split décrits dans `meta_v4.json`, évalue v4 **et** v5 sur le même jeu de test, puis sauvegarde les artefacts v5 sans toucher à v4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Configuration (à adapter)

In [ ]:
BASE      = '/content/drive/MyDrive/nids'
DATA_PATH = f'{BASE}/data/cicids2017_nfstream.parquet'   # .parquet ou .csv
LABEL_COL = 'label'
TIME_COL  = 'bidirectional_first_seen_ms'                # temps en millisecondes
LABEL_MAP = {}   # ex : {'DoS slowloris': 'DoS-slowloris', 'Web Attack': 'WebAttack'}

OUT_VERSION       = 'v5'
SEEDS             = [0, 1, 2]
CAP_TRAIN         = 40000      # plafond par classe (comme v4)
USE_CLASS_WEIGHTS = False      # v4 : aucun. Mettre True pour tester des poids 1/sqrt(n)
DROPOUT           = 0.2        # v4 inconnu : valeur de départ à ajuster
LR, BATCH, MAX_EPOCHS, PATIENCE = 1e-3, 1024, 60, 8

## 2. Chargement des métadonnées et du dataset

In [ ]:
import json, sys, os, joblib, numpy as np, pandas as pd, torch, torch.nn as nn, sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix

CLASSES  = json.load(open(f'{BASE}/classes.json'))
FEATURES = json.load(open(f'{BASE}/feature_cols.json'))
META_V4  = json.load(open(f'{BASE}/meta_v4.json'))
cls2id   = {c: i for i, c in enumerate(CLASSES)}
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device :', dev, '| classes :', len(CLASSES), '| features :', len(FEATURES))

# Versions : comparer avec env_v4.json (important pour recharger scaler_v4 et dnn_v4)
print('Actuel :', dict(python=sys.version.split()[0], torch=torch.__version__,
                       sklearn=sklearn.__version__, numpy=np.__version__, pandas=pd.__version__))
print('env_v4 :', json.load(open(f'{BASE}/env_v4.json')))

df = pd.read_parquet(DATA_PATH) if DATA_PATH.endswith('.parquet') else pd.read_csv(DATA_PATH)
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().replace(LABEL_MAP)
print('\nÉtiquettes hors des 12 classes (exclues) :')
print(df.loc[~df[LABEL_COL].isin(CLASSES), LABEL_COL].value_counts().to_string() or 'aucune')

missing = [c for c in FEATURES + [TIME_COL] if c not in df.columns]
assert not missing, f'Colonnes absentes du dataset : {missing}'
df = df[df[LABEL_COL].isin(CLASSES)].dropna(subset=[TIME_COL]).reset_index(drop=True)
print('\nFlux conservés :', len(df))
print(df[LABEL_COL].value_counts().to_string())

## 3. Split identique à v4
Attaques : coupe chronologique 60/20/20 par classe, avec marge de 1 % entre les parties. Normal : blocs de 10 min tirés au hasard.
*Hypothèse :* l'implémentation exacte de v4 n'est pas connue, donc ce test peut différer légèrement de celui de `meta_v4.json`. C'est pourquoi v4 est réévalué ici.

In [ ]:
rng = np.random.RandomState(0)
t = df[TIME_COL].values.astype(np.float64)
lab = df[LABEL_COL].values
y_all = np.array([cls2id[l] for l in lab])
split = np.full(len(df), '', dtype=object)

for c in CLASSES:
    idx = np.where(lab == c)[0]
    if c == 'BENIGN':
        blocks = (t[idx] // 600000).astype(np.int64)         # blocs de 10 min
        ub = np.unique(blocks); rng.shuffle(ub)
        n = len(ub); a, b = int(.6 * n), int(.8 * n)
        m = {k: 'train' for k in ub[:a]}
        m.update({k: 'val' for k in ub[a:b]}); m.update({k: 'test' for k in ub[b:]})
        split[idx] = [m[k] for k in blocks]
    else:
        o = idx[np.argsort(t[idx], kind='stable')]
        n = len(o)
        split[o[:int(.59 * n)]] = 'train'
        split[o[int(.60 * n):int(.79 * n)]] = 'val'
        split[o[int(.80 * n):]] = 'test'

tr_all = np.where(split == 'train')[0]
keep = []
for c in CLASSES:
    ii = tr_all[lab[tr_all] == c]
    if len(ii) > CAP_TRAIN:
        ii = np.random.RandomState(0).choice(ii, CAP_TRAIN, replace=False)
    keep.append(ii)
tr_idx = np.concatenate(keep)
va_idx = np.where(split == 'val')[0]
te_idx = np.where(split == 'test')[0]

print(pd.DataFrame({'train': pd.Series(lab[tr_idx]).value_counts(),
                    'val':   pd.Series(lab[va_idx]).value_counts(),
                    'test':  pd.Series(lab[te_idx]).value_counts()}).reindex(CLASSES).fillna(0).astype(int))

## 4. Prétraitement (identique à v4) et scaler v5

In [ ]:
def prep(X):
    # inf/-inf -> NaN -> 0 ; clip(min=0) ; log1p   (le StandardScaler est appliqué ensuite)
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
    return np.log1p(X.values.astype(np.float64)).astype(np.float32)

Xtr_raw = prep(df.iloc[tr_idx][FEATURES])
scaler_v5 = StandardScaler().fit(Xtr_raw)            # ajusté sur le train uniquement
Xtr = scaler_v5.transform(Xtr_raw).astype(np.float32)
Xva = scaler_v5.transform(prep(df.iloc[va_idx][FEATURES])).astype(np.float32)
Xte_raw = prep(df.iloc[te_idx][FEATURES])
Xte = scaler_v5.transform(Xte_raw).astype(np.float32)
ytr, yva, yte = y_all[tr_idx], y_all[va_idx], y_all[te_idx]
print(Xtr.shape, Xva.shape, Xte.shape)

## 5. Modèle : architecture lue dans `dnn_v4.pt`
Le notebook infère les couches (Linear / BatchNorm) depuis les poids de v4 pour que v5 ait la même structure. Si une assertion échoue, collez ici la définition de votre modèle.

In [ ]:
def load_ckpt(path):
    obj = torch.load(path, map_location='cpu', weights_only=False)   # vos propres fichiers uniquement
    if isinstance(obj, nn.Module):
        return obj, obj.state_dict(), 'module'
    if isinstance(obj, dict):
        for k in ('state_dict', 'model_state_dict', 'model'):
            if k in obj and isinstance(obj[k], dict):
                return obj, obj[k], 'wrapped:' + k
    return obj, obj, 'state_dict'

v4_obj, v4_sd, v4_kind = load_ckpt(f'{BASE}/dnn_v4.pt')
print('Format de dnn_v4.pt :', v4_kind)
for k, v in v4_sd.items():
    print(f'{k:40s} {tuple(v.shape)}')

def infer_arch(sd):
    arch = []
    for k, v in sd.items():
        if k.endswith('num_batches_tracked'): continue
        if k.endswith('weight') and v.ndim == 2: arch.append(['linear', v.shape[1], v.shape[0]])
        elif k.endswith('running_mean'):          arch.append(['bn', v.shape[0]])
    return arch

def build_model(arch, dropout=0.0):
    mods = []
    last = max(i for i, l in enumerate(arch) if l[0] == 'linear')
    drop = [nn.Dropout(dropout)] if dropout else []
    for i, l in enumerate(arch):
        if l[0] == 'linear':
            mods.append(nn.Linear(l[1], l[2]))
            nxt = arch[i + 1][0] if i + 1 < len(arch) else None
            if i != last and nxt != 'bn':
                mods += [nn.ReLU()] + drop
        else:
            mods += [nn.BatchNorm1d(l[1]), nn.ReLU()] + drop
    return nn.Sequential(*mods)

ARCH = infer_arch(v4_sd)
print('\nArchitecture inférée :', ARCH)
assert ARCH[0][1] == len(FEATURES), f'entrée {ARCH[0][1]} != {len(FEATURES)} features'
assert [l for l in ARCH if l[0] == 'linear'][-1][2] == len(CLASSES), 'sortie != nombre de classes'

def load_by_position(model, sd):
    src = [v for k, v in sd.items() if not k.endswith('num_batches_tracked')]
    dst = [k for k in model.state_dict() if not k.endswith('num_batches_tracked')]
    assert len(src) == len(dst), f'{len(src)} tenseurs source, {len(dst)} attendus'
    ref = model.state_dict()
    new = {}
    for k, v in zip(dst, src):
        assert ref[k].shape == v.shape, f'forme différente pour {k}'
        new[k] = v
    model.load_state_dict(new, strict=False)
    return model

@torch.no_grad()
def predict(model, X, bs=8192):
    model.eval().to(dev)
    out = []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i + bs]).to(dev)
        out.append(model(xb).argmax(1).cpu().numpy())
    return np.concatenate(out)

def metrics(y_true, pred):
    ben = cls2id['BENIGN']
    return dict(F1_macro=round(f1_score(y_true, pred, average='macro'), 4),
                attaques_ratees=int(((y_true != ben) & (pred == ben)).sum()),
                fausses_alertes=int(((y_true == ben) & (pred != ben)).sum()))

## 6. Référence : v4 sur CE jeu de test

In [ ]:
v4_model = build_model(ARCH, 0.0)
load_by_position(v4_model, v4_sd)
scaler_v4 = joblib.load(f'{BASE}/scaler_v4.joblib')
Xte_v4 = scaler_v4.transform(Xte_raw).astype(np.float32)
pred_v4 = predict(v4_model, Xte_v4)
res_v4 = metrics(yte, pred_v4)
print('v4 sur ce test :', res_v4)
print('v4 dans meta_v4.json :', META_V4['resultats_test_3_graines_sans_poids'])

## 7. Entraînement v5 (3 graines)

In [ ]:
def train_one(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = build_model(ARCH, DROPOUT).to(dev)
    w = None
    if USE_CLASS_WEIGHTS:
        cnt = np.bincount(ytr, minlength=len(CLASSES)).astype(np.float64)
        w = 1.0 / np.sqrt(np.maximum(cnt, 1)); w = torch.tensor(w / w.mean(), dtype=torch.float32, device=dev)
    lossf = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    Xt = torch.from_numpy(Xtr).to(dev); yt = torch.from_numpy(ytr).long().to(dev)
    best, best_state, bad = -1, None, 0
    for ep in range(MAX_EPOCHS):
        model.train()
        perm = torch.randperm(len(Xt), device=dev)
        for i in range(0, len(Xt), BATCH):
            ib = perm[i:i + BATCH]
            if len(ib) < 2: continue            # BatchNorm exige > 1 exemple
            opt.zero_grad(); lossf(model(Xt[ib]), yt[ib]).backward(); opt.step()
        f1 = f1_score(yva, predict(model, Xva), average='macro')
        if f1 > best:
            best, bad = f1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: break
    model.load_state_dict(best_state)
    print(f'graine {seed} : arrêt époque {ep + 1}, meilleur F1 val = {best:.4f}')
    return model, best_state

runs = {}
for s in SEEDS:
    m, sd = train_one(s)
    pred = predict(m, Xte)
    runs[s] = dict(model=m, sd=sd, pred=pred, **metrics(yte, pred))
    print('   test :', {k: runs[s][k] for k in ('F1_macro', 'attaques_ratees', 'fausses_alertes')})

## 8. Comparaison v4 / v5

In [ ]:
rows = [dict(modele='v4', graine='-', **res_v4)]
rows += [dict(modele=OUT_VERSION, graine=s, **{k: r[k] for k in ('F1_macro', 'attaques_ratees', 'fausses_alertes')})
         for s, r in runs.items()]
print(pd.DataFrame(rows).to_string(index=False))

s0 = SEEDS[0]
print('\n=== v4 : rapport par classe ===')
print(classification_report(yte, pred_v4, labels=range(len(CLASSES)), target_names=CLASSES, digits=3, zero_division=0))
print(f'=== {OUT_VERSION} (graine {s0}) : rapport par classe ===')
print(classification_report(yte, runs[s0]['pred'], labels=range(len(CLASSES)), target_names=CLASSES, digits=3, zero_division=0))
print('Matrice de confusion (lignes = vrai, colonnes = prédit) :')
print(pd.DataFrame(confusion_matrix(yte, runs[s0]['pred'], labels=range(len(CLASSES))),
                   index=CLASSES, columns=[c[:8] for c in CLASSES]))

## 9. Sauvegarde de v5 (v4 n'est jamais modifié)
Les clés du `state_dict` sont recalées sur celles de `dnn_v4.pt` pour que votre code de chargement actuel fonctionne sans changement. **Ne déployez v5 que s'il bat v4 sur ce tableau, notamment sur les attaques ratées et les fausses alertes.**

In [ ]:
import platform
OUT = BASE
sd5 = runs[SEEDS[0]]['sd']            # graine 0 : pas de sélection sur le test

if v4_kind == 'state_dict':
    v4_keys = list(v4_sd.keys())
    v5_vals = list(sd5.values())
    assert len(v4_keys) == len(v5_vals), 'structure différente de v4'
    to_save = dict(zip(v4_keys, v5_vals))
    torch.save(to_save, f'{OUT}/dnn_{OUT_VERSION}.pt')
else:
    print('ATTENTION : dnn_v4.pt est de type', v4_kind, '; v5 est sauvegardé en state_dict (Sequential).')
    torch.save(sd5, f'{OUT}/dnn_{OUT_VERSION}.pt')

joblib.dump(scaler_v5, f'{OUT}/scaler_{OUT_VERSION}.joblib')

meta5 = dict(META_V4)
meta5['version'] = OUT_VERSION
meta5['scaler'] = f'scaler_{OUT_VERSION}.joblib'
meta5['reglage'] = {'poids_classes': 'sqrt-inverse' if USE_CLASS_WEIGHTS else 'aucun',
                    'graine': SEEDS[0], 'dropout': DROPOUT, 'lr': LR, 'batch': BATCH,
                    'epoques_max': MAX_EPOCHS, 'patience': PATIENCE}
meta5['plafond_train_par_classe'] = CAP_TRAIN
meta5['architecture'] = ARCH
meta5['resultats_test_sans_poids' if not USE_CLASS_WEIGHTS else 'resultats_test_avec_poids'] = {
    'F1_macro': [runs[s]['F1_macro'] for s in SEEDS],
    'attaques_ratees': [runs[s]['attaques_ratees'] for s in SEEDS],
    'fausses_alertes': [runs[s]['fausses_alertes'] for s in SEEDS]}
meta5['reference_v4_meme_test'] = res_v4
json.dump(meta5, open(f'{OUT}/meta_{OUT_VERSION}.json', 'w'), indent=2, ensure_ascii=False)

json.dump(dict(python=platform.python_version(), torch=torch.__version__, sklearn=sklearn.__version__,
               numpy=np.__version__, pandas=pd.__version__, joblib=joblib.__version__),
          open(f'{OUT}/env_{OUT_VERSION}.json', 'w'), indent=2)
print('Fichiers écrits dans', OUT, ':', sorted(f for f in os.listdir(OUT) if OUT_VERSION in f))